<a href="https://colab.research.google.com/github/kevinl03/stochastic-spread-modeling/blob/migrate-statarb-work/statarb/cex_gbm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# StatArb LightGBM Pipeline
Cross-exchange spread z-score prediction across all coins × exchange pairs.

**Target:** z-score of `spread_pct` at `t+5` snapshots, rolling-normalised over 60 snapshots  
**Features:** lag-1..5 of spread, ticker mid/BA, orderbook imbalance, trade flow, funding rate, OI  
**Model:** single LightGBM; `coin` and `pair` as native categoricals

## 1. Imports & Config

In [13]:
import json
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from dotenv import load_dotenv
import os

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

# ── paths — adjust if your layout differs ────────────────────────────────────
DATA_ROOT  = Path("./cex_dat")  # unused if USE_HF = True
USE_HF     = True               # ← add this
HF_REPO    = "SFU-fintech-AI/statarb-crypto-research"

OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── model / feature config ────────────────────────────────────────────────────
HORIZON       = 5    # snapshots ahead to predict (~5 min)
ZSCORE_WINDOW = 60   # rolling window for z-score normalisation (default was 60 min)
N_LAGS        = 5    # lag depth for every feature group
MIN_PERIODS   = 20   # min observations before z-score is emitted

LGBM_PARAMS = {
    "objective":         "regression",
    "metric":            ["rmse", "mae"],
    "learning_rate":     0.05,
    "num_leaves":        127,
    "min_child_samples": 50,
    "feature_fraction":  0.7,
    "bagging_fraction":  0.8,
    "bagging_freq":      5,
    "lambda_l1":         0.1,
    "lambda_l2":         0.1,
    "verbosity":         -1,
    "n_jobs":            -1,
    "seed":              42,
}
NUM_BOOST_ROUND = 1000
EARLY_STOPPING  = 50

In [14]:
TOP_EXCHANGES = ["binance", "bybit", "okx", "coinbase", "kraken"]  # drop smaller venues
TOP_FEATURES  = {
    "ticker":    ["mid", "spread_bps", "bid_volume", "ask_volume"],  # drop vwap, pct_change_24h
    "orderbook": ["imbalance", "slippage_bps"],                      # drop depth/vwap cols
    "trades":    ["buy_sell_ratio", "total_volume"],                  # drop count, price_mean
    "funding":   ["funding_rate"],                                    # drop next_rate, mark_price
    "oi":        ["oi_amount"],                                       # drop oi_value (mostly null)
}

## 2. Data Loading
Drops error rows (`error IS NOT NULL`) on load — these are failed ccxt fetches, not missing data.

In [15]:
SUBSETS = ["spread_matrix", "ticker", "orderbook", "trades", "funding_rate", "open_interest"]

"""
def load_parquet(split_dir: Path, name: str) -> pd.DataFrame:
    path = split_dir / f"{name}.parquet"
    if not path.exists():
        print(f"  [skip] {path.name} not found")
        return pd.DataFrame()
    df = pd.read_parquet(path)
    if "error" in df.columns:
        df = df[df["error"].isna()].drop(columns=["error"])
    return df

def load_split(split_dir: Path) -> dict[str, pd.DataFrame]:
    return {n: load_parquet(split_dir, n) for n in SUBSETS}

print("Loading splits …")
train_raw = load_split(DATA_ROOT / "1_train")
test_raw  = load_split(DATA_ROOT / "2_test")
val_raw   = load_split(DATA_ROOT / "3_val")

for split, raw in [("train", train_raw), ("test", test_raw), ("val", val_raw)]:
    print(f"\n{split}:")
    for k, v in raw.items():
        print(f"  {k:25s} {len(v):>10,} rows")
"""

# HuggingFace subset names per split
HF_SUBSETS = {
    "train": {
        "spread_matrix": "spread_matrix",
        "ticker":        "ticker",
        "orderbook":     "orderbook",
        "trades":        "trades",
        "funding_rate":  "funding_rate",
        "open_interest": "open_interest",
    },
    "test": {
        "spread_matrix": "test_spread_matrix",
        "ticker":        "test_ticker",
        "orderbook":     "test_orderbook",
        "trades":        "test_trades",
        "funding_rate":  "test_funding_rate",
        "open_interest": "test_open_interest",
    },
    "val": {
        "spread_matrix": "validation_spread_matrix",
        "ticker":        "validation_ticker",
        "orderbook":     "validation_orderbook",
        "trades":        "validation_trades",
        "funding_rate":  "validation_funding_rate",
        "open_interest": "validation_open_interest",
    },
}

def load_split(split_name: str) -> dict[str, pd.DataFrame]:
    from datasets import load_dataset
    result = {}
    for local_name, hf_name in HF_SUBSETS[split_name].items():
        print(f"  loading {hf_name} …", end=" ")
        df = load_dataset(HF_REPO, hf_name, split="train").to_pandas()
        if "error" in df.columns:
            df = df[df["error"].isna()].drop(columns=["error"])
        print(df.shape)
        result[local_name] = df
    return result

train_raw = load_split("train")
test_raw  = load_split("test")
val_raw   = load_split("val")

  loading spread_matrix … 

DatasetNotFoundError: Dataset 'SFU-fintech-AI/statarb-crypto-research' doesn't exist on the Hub or cannot be accessed.

## 3. Feature Engineering — Spread Matrix
This is the backbone. Every other feature table joins onto the `(snapshot_idx, coin, pair)` grain produced here.

- Parses `payload` JSON if columns aren't already flat
- Derives `spread_pct` from whatever price columns are present
- Computes rolling z-score **and** shifts it forward by `HORIZON` to form the target
- Adds lag-1..N of both `spread_pct` and `zscore`

In [ ]:
def build_spread_features(sm: pd.DataFrame) -> pd.DataFrame:
    if sm.empty:
        return pd.DataFrame()

    # ── explode pairwise_spreads list from payload ────────────────────────
    records = []
    for row in sm.itertuples():
        try:
            p = json.loads(row.payload) if isinstance(row.payload, str) else row.payload
            for pair in p["pairwise_spreads"]:
                records.append({
                    "snapshot_idx": row.snapshot_idx,
                    "coin":         row.coin,
                    "exchange_a":   pair["ex1"],
                    "exchange_b":   pair["ex2"],
                    "spread_bps":   pair["spread_bps"],
                })
        except Exception as e:
            continue  # skip malformed rows silently

    sm = pd.DataFrame(records)
    if sm.empty:
        return pd.DataFrame()

    sm["pair"] = sm["exchange_a"] + "__" + sm["exchange_b"]
    sm = sm.sort_values(["coin", "pair", "snapshot_idx"]).reset_index(drop=True)

    # ── z-score per (coin, pair), target = z-score at t+HORIZON ─────────
    grp = sm.groupby(["coin", "pair"])["spread_bps"]
    roll_mean = grp.transform(lambda x: x.rolling(ZSCORE_WINDOW, min_periods=MIN_PERIODS).mean())
    roll_std  = grp.transform(lambda x: x.rolling(ZSCORE_WINDOW, min_periods=MIN_PERIODS).std())
    sm["zscore"] = (sm["spread_bps"] - roll_mean) / roll_std.replace(0, np.nan)
    sm["target"] = sm.groupby(["coin", "pair"])["zscore"].transform(lambda x: x.shift(-HORIZON))

    # ── lag features ──────────────────────────────────────────────────────
    for lag in range(1, N_LAGS + 1):
        sm[f"spread_bps_lag{lag}"] = grp.transform(lambda x, l=lag: x.shift(l))
        sm[f"zscore_lag{lag}"] = sm.groupby(["coin", "pair"])["zscore"].transform(
            lambda x, l=lag: x.shift(l)
        )

    return sm

## 4. Feature Engineering — Ticker
Derives mid-price and bid-ask spread (%), lags both, then pivots wide so each exchange becomes its own column set.

In [ ]:
def build_ticker_features(tk: pd.DataFrame) -> pd.DataFrame:
    if tk.empty:
        return pd.DataFrame()

    records = []
    for row in tk.itertuples():
        try:
            p = json.loads(row.payload) if isinstance(row.payload, str) else row.payload
            records.append({
                "snapshot_idx": row.snapshot_idx,
                "coin":         row.coin,
                "exchange":     row.exchange,
                "mid":          p["mid"],
                "spread_bps":   p["spread_bps"],
                "bid_volume":   p["bid_volume"],
                "ask_volume":   p["ask_volume"],
                "pct_change_24h": p.get("pct_change_24h"),
                "vwap":         p.get("vwap"),
            })
        except Exception:
            continue

    tk = pd.DataFrame(records).sort_values(["coin", "exchange", "snapshot_idx"])
    tk = tk[tk["exchange"].isin(TOP_EXCHANGES)]
    num_cols = tk.select_dtypes(include="float64").columns
    tk[num_cols] = tk[num_cols].astype("float32")

    feat_cols = TOP_FEATURES["ticker"]
    for col in feat_cols:
        for lag in range(1, N_LAGS + 1):
            tk[f"{col}_lag{lag}"] = tk.groupby(["coin", "exchange"])[col].transform(
                lambda x, l=lag: x.shift(l)
            )

    lag_cols = [f"{col}_lag{lag}" for col in feat_cols for lag in range(1, N_LAGS + 1)]
    wide = tk.pivot_table(index=["snapshot_idx", "coin"], columns="exchange",
                          values=lag_cols, aggfunc="first")
    wide.columns = [f"tk_{col}_{exch}" for col, exch in wide.columns]
    return wide.reset_index()


## 5. Feature Engineering — Orderbook
Computes top-5-level order book imbalance: `(bid_vol - ask_vol) / (bid_vol + ask_vol)`.  
Parses nested `bids`/`asks` lists from payload if volume columns aren't already present.

In [ ]:
def build_orderbook_features(ob: pd.DataFrame) -> pd.DataFrame:
    if ob.empty:
        return pd.DataFrame()

    records = []
    for row in ob.itertuples():
        try:
            p = json.loads(row.payload) if isinstance(row.payload, str) else row.payload
            records.append({
                "snapshot_idx":   row.snapshot_idx,
                "coin":           row.coin,
                "exchange":       row.exchange,
                "imbalance":      p["imbalance"],
                "slippage_bps":   p["slippage_bps"],
                "bid_depth":      p["bid_depth_units"],
                "ask_depth":      p["ask_depth_units"],
                "bid_vwap":       p["bid_vwap"],
                "ask_vwap":       p["ask_vwap"],
            })
        except Exception:
            continue

    ob = pd.DataFrame(records).sort_values(["coin", "exchange", "snapshot_idx"])
    ob = ob[ob["exchange"].isin(TOP_EXCHANGES)]
    num_cols = ob.select_dtypes(include="float64").columns
    ob[num_cols] = ob[num_cols].astype("float32")

    feat_cols = TOP_FEATURES["orderbook"]
    for col in feat_cols:
        for lag in range(1, N_LAGS + 1):
            ob[f"{col}_lag{lag}"] = ob.groupby(["coin", "exchange"])[col].transform(
                lambda x, l=lag: x.shift(l)
            )

    lag_cols = [f"{col}_lag{lag}" for col in feat_cols for lag in range(1, N_LAGS + 1)]
    wide = ob.pivot_table(index=["snapshot_idx", "coin"], columns="exchange",
                          values=lag_cols, aggfunc="first")
    wide.columns = [f"ob_{col}_{exch}" for col, exch in wide.columns]
    return wide.reset_index()


## 6. Feature Engineering — Trades
Net signed flow per snapshot: `sum(buy_vol) - sum(sell_vol)`.  
If `side` is missing, falls back to unsigned total volume.

In [ ]:
def build_trades_features(tr: pd.DataFrame) -> pd.DataFrame:
    if tr.empty:
        return pd.DataFrame()

    records = []
    for row in tr.itertuples():
        try:
            p = json.loads(row.payload) if isinstance(row.payload, str) else row.payload
            records.append({
                "snapshot_idx":   row.snapshot_idx,
                "coin":           row.coin,
                "exchange":       row.exchange,
                "buy_sell_ratio": p["buy_sell_ratio"],
                "buy_volume":     p["buy_volume"],
                "sell_volume":    p["sell_volume"],
                "total_volume":   p["total_volume"],
                "trade_count":    p["trade_count"],
                "price_mean":     p["price_mean"],
            })
        except Exception:
            continue

    tr = pd.DataFrame(records).sort_values(["coin", "exchange", "snapshot_idx"])
    tr = tr[tr["exchange"].isin(TOP_EXCHANGES)]
    num_cols = tr.select_dtypes(include="float64").columns
    tr[num_cols] = tr[num_cols].astype("float32")

    feat_cols = TOP_FEATURES["trades"]
    for col in feat_cols:
        for lag in range(1, N_LAGS + 1):
            tr[f"{col}_lag{lag}"] = tr.groupby(["coin", "exchange"])[col].transform(
                lambda x, l=lag: x.shift(l)
            )

    lag_cols = [f"{col}_lag{lag}" for col in feat_cols for lag in range(1, N_LAGS + 1)]
    wide = tr.pivot_table(index=["snapshot_idx", "coin"], columns="exchange",
                          values=lag_cols, aggfunc="first")
    wide.columns = [f"tr_{col}_{exch}" for col, exch in wide.columns]
    return wide.reset_index()


## 7. Feature Engineering — Funding Rate & Open Interest
Both are perp-only signals (NaN for spot venues). LightGBM handles NaN natively so no imputation needed.

In [ ]:
def build_funding_features(fr: pd.DataFrame) -> pd.DataFrame:
    if fr.empty:
        return pd.DataFrame()

    records = []
    for row in fr.itertuples():
        try:
            p = json.loads(row.payload) if isinstance(row.payload, str) else row.payload
            records.append({
                "snapshot_idx":      row.snapshot_idx,
                "coin":              row.coin,
                "exchange":          row.exchange,
                "funding_rate":      p["funding_rate"],
                "next_funding_rate": p.get("next_funding_rate"),
                "mark_price":        p.get("mark_price"),
            })
        except Exception:
            continue

    fr = pd.DataFrame(records).sort_values(["coin", "exchange", "snapshot_idx"])
    fr = fr[fr["exchange"].isin(TOP_EXCHANGES)]
    num_cols = fr.select_dtypes(include="float64").columns
    fr[num_cols] = fr[num_cols].astype("float32")

    feat_cols = TOP_FEATURES["funding"]
    for col in feat_cols:
        for lag in range(1, N_LAGS + 1):
            fr[f"{col}_lag{lag}"] = fr.groupby(["coin", "exchange"])[col].transform(
                lambda x, l=lag: x.shift(l)
            )

    lag_cols = [f"{col}_lag{lag}" for col in feat_cols for lag in range(1, N_LAGS + 1)]
    wide = fr.pivot_table(index=["snapshot_idx", "coin"], columns="exchange",
                          values=lag_cols, aggfunc="first")
    wide.columns = [f"fr_{col}_{exch}" for col, exch in wide.columns]
    return wide.reset_index()



def build_oi_features(oi: pd.DataFrame) -> pd.DataFrame:
    if oi.empty:
        return pd.DataFrame()

    records = []
    for row in oi.itertuples():
        try:
            p = json.loads(row.payload) if isinstance(row.payload, str) else row.payload
            records.append({
                "snapshot_idx": row.snapshot_idx,
                "coin":         row.coin,
                "exchange":     row.exchange,
                "oi_amount":    p["open_interest_amount"],
                "oi_value":     p.get("open_interest_value"),
            })
        except Exception:
            continue

    oi = pd.DataFrame(records).sort_values(["coin", "exchange", "snapshot_idx"])
    oi = oi[oi["exchange"].isin(TOP_EXCHANGES)]
    num_cols = oi.select_dtypes(include="float64").columns
    oi[num_cols] = oi[num_cols].astype("float32")

    feat_cols = TOP_FEATURES["oi"]
    for col in feat_cols:
        for lag in range(1, N_LAGS + 1):
            oi[f"{col}_lag{lag}"] = oi.groupby(["coin", "exchange"])[col].transform(
                lambda x, l=lag: x.shift(l)
            )

    lag_cols = [f"{col}_lag{lag}" for col in feat_cols for lag in range(1, N_LAGS + 1)]
    wide = oi.pivot_table(index=["snapshot_idx", "coin"], columns="exchange",
                          values=lag_cols, aggfunc="first")
    wide.columns = [f"oi_{col}_{exch}" for col, exch in wide.columns]
    return wide.reset_index()

## 8. Merge All Features
Left-join everything onto the spread base frame on `(snapshot_idx, coin)`.  
Missing auxiliary rows become NaN — handled by LightGBM natively.

In [ ]:
def build_feature_matrix(raw: dict[str, pd.DataFrame]) -> pd.DataFrame:
    print("  spread …", end=" ")
    base = build_spread_features(raw["spread_matrix"])
    if base.empty:
        raise ValueError("spread_matrix empty or unparseable")
    print(f"{base.shape}")

    aux_builders = {
        "ticker":   (build_ticker_features,   raw["ticker"]),
        "orderbook":(build_orderbook_features, raw["orderbook"]),
        "trades":   (build_trades_features,    raw["trades"]),
        "funding":  (build_funding_features,   raw["funding_rate"]),
        "OI":       (build_oi_features,        raw["open_interest"]),
    }

    df = base.copy()
    for label, (fn, data) in aux_builders.items():
        aux = fn(data)
        if aux is not None and not aux.empty:
            df = df.merge(aux, on=["snapshot_idx", "coin"], how="left")
            print(f"  + {label:10s} → {df.shape}")

    return df

print("Building train …")
df_train = build_feature_matrix(train_raw)
print("\nBuilding test …")
df_test  = build_feature_matrix(test_raw)
print("\nBuilding val …")
df_val   = build_feature_matrix(val_raw)

## 9. Prepare LightGBM Datasets
- Drops rows where `target` is NaN (first `HORIZON` rows of each group, and z-score warmup rows)
- `coin` and `pair` are encoded as LightGBM native categoricals — no one-hot needed
- Test/val columns are re-aligned to train's column set (some exchanges may be absent in shorter splits)

In [ ]:
ID_COLS = {"snapshot_idx", "exchange_a", "exchange_b", "spread_bps", "zscore", "target", "p1", "p2"}

def prepare_dataset(df: pd.DataFrame, reference_cols=None):
    df = df.dropna(subset=["target"]).copy()

    cat_cols = [c for c in ["coin", "pair"] if c in df.columns]
    for c in cat_cols:
        df[c] = df[c].astype("category")

    feat_cols = [c for c in df.columns if c not in ID_COLS]
    X = df[feat_cols].copy()
    y = df["target"].values

    if reference_cols is not None:
        X = X.reindex(columns=reference_cols)
        for c in cat_cols:
            if c in X.columns:
                X[c] = X[c].astype("category")

    return X, y, feat_cols, cat_cols

X_train, y_train, feat_cols, cat_cols = prepare_dataset(df_train)
X_test,  y_test,  _,         _        = prepare_dataset(df_test,  reference_cols=X_train.columns)
X_val,   y_val,   _,         _        = prepare_dataset(df_val,   reference_cols=X_train.columns)

print(f"Train : {X_train.shape}  |  target mean={y_train.mean():.3f}  std={y_train.std():.3f}")
print(f"Test  : {X_test.shape}")
print(f"Val   : {X_val.shape}")

## 10. Train
Early stopping is evaluated on the **test** split.  
Once architecture is locked, consider using a time-based slice of train for early stopping instead, so test stays fully held-out.

In [ ]:
dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols, free_raw_data=False)
dtest  = lgb.Dataset(X_test,  label=y_test,  categorical_feature=cat_cols,
                     reference=dtrain, free_raw_data=False)

model = lgb.train(
    LGBM_PARAMS,
    dtrain,
    num_boost_round=NUM_BOOST_ROUND,
    valid_sets=[dtrain, dtest],
    valid_names=["train", "test"],
    callbacks=[
        lgb.early_stopping(EARLY_STOPPING, verbose=True),
        lgb.log_evaluation(100),
    ],
)
print(f"\nBest iteration: {model.best_iteration}")

## 11. Evaluate
`dir_acc` (directional accuracy) is the most operationally meaningful metric —  
it tells you how often the model correctly predicts whether the spread is above or below its rolling mean.

In [ ]:
def evaluate(model, X, y, label):
    preds   = model.predict(X, num_iteration=model.best_iteration)
    mae     = mean_absolute_error(y, preds)
    rmse    = mean_squared_error(y, preds, squared=False)
    r2      = r2_score(y, preds)
    dir_acc = np.mean(np.sign(preds) == np.sign(y))
    print(f"{label:20s}  MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}  DirAcc={dir_acc:.3%}")
    return {"label": label, "mae": mae, "rmse": rmse, "r2": r2, "dir_acc": dir_acc}

results = []
results.append(evaluate(model, X_train, y_train, "train"))
results.append(evaluate(model, X_test,  y_test,  "test"))
if len(y_val) > 0:
    results.append(evaluate(model, X_val, y_val, "val (partial)"))

results_df = pd.DataFrame(results)
results_df

## 12. Feature Importance

In [ ]:
imp_df = pd.DataFrame({
    "feature":    model.feature_name(),
    "importance": model.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=False).reset_index(drop=True)

print(imp_df.head(30).to_string())

## 13. Save Outputs

In [ ]:
model.save_model(str(OUTPUT_DIR / "statarb_lgbm.txt"))
imp_df.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)
results_df.to_csv(OUTPUT_DIR / "eval_results.csv", index=False)

print(f"Saved to {OUTPUT_DIR}/")
print("  statarb_lgbm.txt")
print("  feature_importance.csv")
print("  eval_results.csv")